In [ ]:
import os
filePath = ''
file_list = os.listdir(filePath)
print(file_list)

In [ ]:
import pandas as pd
import tqdm
import csv
ICEWS_all_entites = set()
#ICEWS_all_stamp_fact = set()

e_2_des = {}

for file_name in tqdm.tqdm(file_list):
    if '.tab' in file_name:
        with open(file_name, 'r') as f:
            text = f.read()
        data = text.split('\n')
        for row in data:
            sperate_row = row.strip().split('\t')
            if len(sperate_row) != 20:
                continue
            time = sperate_row[1]
            if time == 'Event Date':
                continue
            s = sperate_row[2]
            s_des = 'Sectors: ' + sperate_row[3] + ' Country:' + sperate_row[4]
            r = sperate_row[5]
            o = sperate_row[8]
            o_des = 'Sectors: ' + sperate_row[9] + ' Country:' + sperate_row[10]
            #all_stamp_fact.add((s, r, o, time))
            ICEWS_all_entites.add(s)
            ICEWS_all_entites.add(o)

            if s not in e_2_des.keys():
                e_2_des[s] = s_des
            if o not in e_2_des.keys():
                e_2_des[o] = o_des
    
    
    elif '.tsv' in file_name:
        with open(file_name, 'r') as f:
            text = f.read()
        data = text.split('\n')
    
        for row in data:
            sperate_row = row.strip().split('\t')
            if len(sperate_row) != 20:
                continue
            time = sperate_row[1]
            if time == 'Event Date':
                continue
            s = sperate_row[2]
            s_des = 'Sectors: ' + sperate_row[3] + ' Country:' + sperate_row[4]
            r = sperate_row[5]
            o = sperate_row[8]
            o_des = 'Sectors: ' + sperate_row[9] + ' Country:' + sperate_row[10]
            #all_stamp_fact.add((s, r, o, time))
            ICEWS_all_entites.add(s)
            ICEWS_all_entites.add(o)

            if s not in e_2_des.keys():
                e_2_des[s] = s_des
            if o not in e_2_des.keys():
                e_2_des[o] = o_des

print(len(ICEWS_all_entites))
#print(len(all_stamp_fact))

#print(e_2_des)
print(list(ICEWS_all_entites)[:10])
#print(list(all_stamp_fact)[:10])
print([(key, e_2_des[key]) for key in list(e_2_des.keys())[:30]])

In [ ]:
# 批量查询Wikidata_id 和文本描述
import requests
import time
import datetime
import csv
import os
import json
import re
from tqdm import tqdm  # 进度条库，如果没有安装请先运行: pip install tqdm
from difflib import SequenceMatcher

class WikidataBatchQuery:
    def __init__(self, output_file, progress_file="progress.json", delay=1.5, batch_size=50):
        """
        初始化Wikidata批量查询
        
        参数:
            output_file: 输出CSV文件路径
            progress_file: 进度保存文件路径
            delay: 批次间延迟（秒）
            batch_size: 每批处理的实体数量
        """
        self.output_file = output_file
        self.progress_file = progress_file
        self.delay = delay
        self.batch_size = batch_size
        
        # 设置请求头
        self.headers = {
            'User-Agent': 'WikidataBatchQuery/1.0 (https://example.com; contact@example.com)',
            'Accept': 'application/json'
        }
        
        # 时间模式正则表达式
        self.time_patterns = [
            r'\(\s*\d{4}\s*[-–—]\s*\d{4}\s*\)',  # (1929-2003)
            r'\(\s*\d{4}\s*\)',  # (1929)
            r'from\s+\d{4}\s+to\s+\d{4}',  # from 1929 to 2003
            r'since\s+\d{4}\s+to\s+\d{4}',  # since 1929 to 2003
            r'\d{4}\s*[-–—]\s*\d{4}',  # 1929-2003 (无括号)
            r'born\s+\d{4}',  # born 1929
            r'\b\d{4}\s*-\s*\d{4}\b',  # 1929 - 2003
            r'\b\d{4}\s*–\s*\d{4}\b',  # 1929 – 2003 (不同种类的横线)
            r'\b\d{4}\s*—\s*\d{4}\b',  # 1929 — 2003
        ]
        
        # 加载进度
        self.processed_entities = self.load_progress()
    
    def clean_description(self, description):
        """
        清理描述文本，移除时间相关的内容
        """
        if not description:
            return ""
        
        cleaned = description
        
        # 应用所有时间模式
        for pattern in self.time_patterns:
            cleaned = re.sub(pattern, '', cleaned, flags=re.IGNORECASE)
        
        # 清理多余的空格和标点
        cleaned = re.sub(r'\s+', ' ', cleaned)  # 多个空格变一个
        cleaned = re.sub(r'\s*,\s*,', ',', cleaned)  # 清理多余的逗号
        cleaned = re.sub(r'^\s*[,-]\s*', '', cleaned)  # 开头是逗号或横线
        cleaned = re.sub(r'\s*[,-]\s*$', '', cleaned)  # 结尾是逗号或横线
        cleaned = cleaned.strip()
        
        return cleaned.replace('()','')
    
    def load_progress(self):
        """加载已处理的实体和进度"""
        processed = set()
        
        # 从输出文件加载已处理的实体
        if os.path.exists(self.output_file):
            with open(self.output_file, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    processed.add(row['original_text'])
        
        # 从进度文件加载（用于断点续传）
        if os.path.exists(self.progress_file):
            try:
                with open(self.progress_file, 'r', encoding='utf-8') as f:
                    progress_data = json.load(f)
                    processed.update(progress_data.get('processed_entities', []))
            except:
                pass
                
        return processed
    
    def save_progress(self, current_batch=None):
        """保存进度到文件"""
        progress_data = {
            'processed_entities': list(self.processed_entities),
            'last_batch': current_batch,
            'timestamp': datetime.datetime.now().isoformat()
        }
        
        with open(self.progress_file, 'w', encoding='utf-8') as f:
            json.dump(progress_data, f, ensure_ascii=False, indent=2)
    
    def batch_wikidata_query(self, entity_batch, max_retries=3):
        """
        批量Wikidata查询函数，一次查询多个实体
        返回：包含(实体, Wikidata ID, 标签, 清理后的描述)的列表
        """
        wikidata_url = "https://www.wikidata.org/w/api.php"
        
        results = []
        
        for entity_text in entity_batch:
            params = {
                'action': 'wbsearchentities',
                'search': entity_text,
                'language': 'en',
                'format': 'json',
                'limit': 5
            }
            try:
                qid, label, description = self.single_entity_query(wikidata_url, params, entity_text, max_retries)
                results.append((entity_text, qid, label, description))
            except:
                pass
        
        return results
    
    def single_entity_query(self, wikidata_url, params, entity_text, max_retries):
        """单个实体的查询逻辑"""
        for attempt in range(max_retries):
            try:
                response = requests.get(wikidata_url, params=params, headers=self.headers, timeout=30)
                
                if response.status_code == 200:
                    data = response.json().get('search', [])
                    if data:
                        best_result = None
                        best_score = 0
                        
                        for result in data:
                            score = SequenceMatcher(None, entity_text.lower(), result['label'].lower()).ratio()
                            score_re = SequenceMatcher(None, result['label'].lower(), entity_text.lower()).ratio()
                            if score > best_score and score > 0.8 and score_re > 0.8:
                                best_score = score
                                best_result = result
                        
                        if best_result is not None:
                            qid = best_result['id']
                            label = best_result['label']
                            description = best_result.get('description', '')
                            cleaned_description = self.clean_description(description)
                            return qid, label, cleaned_description
                        else:
                            return None, None, None
                    return None, None, None
                    
                elif response.status_code == 429:
                    retry_after = response.headers.get('retry-after', 10)
                    timeout = self.get_delay(retry_after)
                    print(f'\n速率限制，等待 {timeout} 秒后重试')
                    time.sleep(timeout)
                    continue
                    
                elif response.status_code == 403:
                    print(f"\n访问被拒绝，等待后重试")
                    if attempt < max_retries - 1:
                        wait_time = (2 ** attempt) * 30
                        print(f"等待 {wait_time} 秒后重试")
                        time.sleep(wait_time)
                    continue
                    
                else:
                    print(f"\nHTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(10)
                        
            except requests.exceptions.RequestException as e:
                print(f"\n请求异常: {e}")
                if attempt < max_retries - 1:
                    time.sleep(10)
        
        return None, None, None
    
    def get_delay(self, date_header):
        """
        从Retry-After头信息计算需要等待的时间
        """
        try:
            date = datetime.datetime.strptime(date_header, '%a, %d %b %Y %H:%M:%S GMT')
            timeout = max(1, int((date - datetime.datetime.now()).total_seconds()))
        except ValueError:
            try:
                timeout = max(1, int(date_header))
            except ValueError:
                timeout = 10
        
        return timeout
    
    def process_entities(self, all_entities):
        """批量处理所有实体"""
        entities = list(all_entities)
        
        print(f"总实体数: {len(entities)}")
        print(f"已处理实体数: {len(self.processed_entities)}")
        print(f"待处理实体数: {len(entities) - len(self.processed_entities)}")
        print(f"批处理大小: {self.batch_size}")
        
        # 过滤已处理的实体
        entities_to_process = [e for e in entities if e not in self.processed_entities]
        
        if not entities_to_process:
            print("所有实体已处理完成!")
            return
        
        # 创建或打开输出文件
        file_exists = os.path.exists(self.output_file)
        with open(self.output_file, 'a', newline='', encoding='utf-8') as f:
            fieldnames = ['original_text', 'wikidata_id', 'wikidata_label', 'short_description']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            
            if not file_exists:
                writer.writeheader()
            
            # 将实体分成批次
            batches = [entities_to_process[i:i + self.batch_size] 
                      for i in range(0, len(entities_to_process), self.batch_size)]
            
            # 使用进度条处理批次
            for batch_num, batch in enumerate(tqdm(batches, desc="处理批次")):
                print(f"\n处理批次 {batch_num + 1}/{len(batches)}, 包含 {len(batch)} 个实体")
                
                # 批量查询
                batch_results = self.batch_wikidata_query(batch)
                
                # 写入批次结果
                for entity_text, qid, label, description in batch_results:
                    writer.writerow({
                        'original_text': entity_text,
                        'wikidata_id': qid if qid else '',
                        'wikidata_label': label if label else '',
                        'short_description': description if description else ''
                    })
                
                f.flush()  # 确保数据写入磁盘
                
                # 更新进度
                self.processed_entities.update(batch)
                
                # 保存进度
                self.save_progress(batch_num)
                
                # 批次间延迟（避免触发API限制）
                if batch_num < len(batches) - 1:  # 最后一批不需要延迟
                    print(f"等待 {self.delay} 秒后处理下一批...")
                    time.sleep(self.delay)
        
        # 处理完成后删除进度文件
        if os.path.exists(self.progress_file):
            os.remove(self.progress_file)
        
        print(f"\n批量查询完成! 结果已保存到: {self.output_file}")

def main():
    # 配置参数
    output_file = "wikidata_results.csv"  # 输出CSV文件
    progress_file = "query_progress.json"  # 进度文件
    
    # 示例实体列表 - 在实际使用中替换为你的实体列表
    all_entities = [
        "Albert Einstein",
        "Marie Curie", 
        "Isaac Newton",
        "Stephen Hawking",
        "Charles Darwin"
        # ... 添加更多实体
    ]
    
    # 创建查询器并执行
    query = WikidataBatchQuery(
        output_file=output_file,
        progress_file=progress_file,
        delay=1.5,  # 批次间延迟
        batch_size=20  # 每批处理的实体数量
    )
    
    # 开始处理
    query.process_entities(all_entities)

# 测试清洗功能的函数
def test_cleaning():
    """测试描述清洗功能"""
    test_cases = [
        "American actor (1929-2003)",
        "American singer from 1929 to 2003",
        "British politician since 1929 to 2003",
        "French painter (1929)",
        "German composer 1929-2003",
        "Italian director born 1929",
        "Spanish writer 1929 – 2003",
        "Portuguese artist 1929—2003",
        "Normal description without dates",
        "Another normal description",
        ""
    ]
    
    query = WikidataBatchQuery("dummy_output.csv")
    
    print("测试描述清洗功能:")
    print("-" * 50)
    for test in test_cases:
        cleaned = query.clean_description(test)
        print(f"原描述: '{test}'")
        print(f"清洗后: '{cleaned}'")
        print("-" * 30)

if __name__ == "__main__":
    # 运行测试
    test_cleaning()

In [ ]:
query = WikidataBatchQuery(
    output_file="results.csv",
    batch_size=50,  # 每批50个实体
    delay=2.0       # 批次间延迟2秒
)

# 处理实体列表
entities = ICEWS_all_entites  # 你的实体列表
query.process_entities(entities)

In [ ]:
# 通过Wikidata_id查询图谱中已有实体的时间区间知识（并行，但返回的是Wikidata_id，需要进一步抽取文本描述和label）
import requests
import json
import csv
import time
import re
import concurrent.futures
from tqdm import tqdm
from collections import defaultdict

class WikidataTemporalPropertiesOptimized:
    def __init__(self, delay=0.1, max_workers=5):
        """
        初始化Wikidata时间属性查询（优化版）
        
        参数:
            delay: 请求间延迟（秒）
            max_workers: 最大并行工作线程数
        """
        self.delay = delay
        self.max_workers = max_workers
        self.headers = {
            'User-Agent': 'WikidataTemporalProperties/2.0 (https://example.com; contact@example.com)'
        }
        
        # 定义时间相关属性
        self.start_time_prop = "P580"  # start time
        self.end_time_prop = "P582"    # end time
        
        # 实体和属性标签缓存
        self.label_cache = {}
        
        # 批量处理相关
        self.batch_size = 50  # 每批处理的实体数量
        
    def get_entities_batch(self, entity_ids, max_retries=3):
        """
        批量获取实体数据
        """
        if not entity_ids:
            return {}
            
        # 将实体ID列表转换为逗号分隔的字符串
        ids_str = "|".join(entity_ids)
        
        url = "https://www.wikidata.org/w/api.php"
        params = {
            'action': 'wbgetentities',
            'ids': ids_str,
            'format': 'json',
            'languages': 'en'
        }
        
        for attempt in range(max_retries):
            try:
                response = requests.get(url, params=params, headers=self.headers, timeout=60)
                
                if response.status_code == 200:
                    data = response.json()
                    return data.get('entities', {})
                elif response.status_code == 429:
                    # 处理速率限制
                    wait_time = 10 * (attempt + 1)
                    print(f"速率限制，等待{wait_time}秒后重试")
                    time.sleep(wait_time)
                    continue
                elif response.status_code == 404:
                    print(f"部分实体不存在")
                    return {}
                else:
                    print(f"HTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
            except requests.exceptions.RequestException as e:
                print(f"请求异常: {e}")
                if attempt < max_retries - 1:
                    time.sleep(5)
        
        return {}
    
    def get_labels_batch(self, entity_ids, max_retries=10):
        """
        批量获取实体标签
        """
        if not entity_ids:
            return {}
            
        # 过滤掉非实体ID
        q_ids = [eid for eid in entity_ids if eid.startswith('Q') or eid.startswith('P')]
        if not q_ids:
            return {}
            
        # 将实体ID列表转换为逗号分隔的字符串
        ids_str = "|".join(q_ids)
        
        url = "https://www.wikidata.org/w/api.php"
        params = {
            'action': 'wbgetentities',
            'ids': ids_str,
            'format': 'json',
            'languages': 'en',
            'props': 'labels'
        }
        
        for attempt in range(max_retries):
                response = requests.get(url, params=params, headers=self.headers, timeout=60)
                print(response)
                if response.status_code == 200:
                    data = response.json()
                    entities = data.get('entities', {})
                    
                    labels = {}
                    for eid, entity in entities.items():
                        labels_data = entity.get('labels', {})
                        if 'en' in labels_data:
                            labels[eid] = labels_data['en'].get('value', eid)
                        else:
                            # 如果没有英文标签，使用第一个可用标签
                            for lang in labels_data:
                                if labels_data[lang].get('value'):
                                    labels[eid] = labels_data[lang].get('value')
                                    break
                            else:
                                labels[eid] = eid
                    
                    # 更新缓存
                    self.label_cache.update(labels)
                    return labels
                elif response.status_code == 429:
                    # 处理速率限制
                    wait_time = 10 * (attempt + 1)
                    print(f"速率限制，等待{wait_time}秒后重试")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"获取标签时HTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
                '''
                except requests.exceptions.RequestException as e:
                    print(f"获取标签时请求异常: {e}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
                '''
        
        return {}
    
    def extract_temporal_claims(self, entity_data):
        """
        从实体数据中提取包含时间信息的声明
        """
        temporal_claims = []
        
        if not entity_data or 'entities' not in entity_data:
            return temporal_claims
        
        # 收集所有需要获取标签的实体ID
        entities_to_label = set()
        
        for entity_id, entity in entity_data['entities'].items():
            # 添加头实体ID到标签查询列表
            entities_to_label.add(entity_id)
            
            if 'claims' not in entity:
                continue
                
            # 遍历所有属性
            for prop_id, claims in entity['claims'].items():
                entities_to_label.add(prop_id)  # 添加属性ID
                
                for claim in claims:
                    # 检查声明是否有限定符
                    if 'qualifiers' in claim:
                        qualifiers = claim['qualifiers']
                        
                        # 检查是否有start time或end time限定符
                        has_start_time = self.start_time_prop in qualifiers
                        has_end_time = self.end_time_prop in qualifiers
                        
                        if has_start_time or has_end_time:
                            # 获取主声明值
                            mainsnak = claim.get('mainsnak', {})
                            datatype = mainsnak.get('datatype', '')
                            datavalue = mainsnak.get('datavalue', {})
                            
                            # 跳过数字类型的值
                            if datatype == 'quantity':
                                continue
                            
                            # 提取主声明值
                            main_value_id, main_value_type = self.extract_main_value_info(mainsnak)
                            if main_value_id and main_value_id.startswith(('Q', 'P')):
                                entities_to_label.add(main_value_id)
                            
                            # 提取时间限定符
                            start_times = self.extract_time_values(qualifiers.get(self.start_time_prop, []))
                            end_times = self.extract_time_values(qualifiers.get(self.end_time_prop, []))
                            
                            temporal_claims.append({
                                'entity_id': entity_id,
                                'property_id': prop_id,
                                'main_value_id': main_value_id,
                                'main_value_type': main_value_type,
                                'start_times': start_times,
                                'end_times': end_times,
                                'has_start_time': has_start_time,
                                'has_end_time': has_end_time,
                                'time_status': self.get_time_status(has_start_time, has_end_time)
                            })
        
        # 批量获取所有需要的标签
        self.get_labels_batch(list(entities_to_label))
        
        # 为每个声明添加标签
        for claim in temporal_claims:
            # 头实体标签
            claim['entity_label'] = self.label_cache.get(claim['entity_id'], claim['entity_id'])
            
            # 属性标签
            claim['property_label'] = self.label_cache.get(claim['property_id'], claim['property_id'])
            
            # 尾实体标签
            main_value_id = claim['main_value_id']
            if main_value_id and main_value_id.startswith(('Q', 'P')):
                claim['main_value_label'] = self.label_cache.get(main_value_id, main_value_id)
            else:
                claim['main_value_label'] = main_value_id
        
        return temporal_claims
    
    def extract_main_value_info(self, mainsnak):
        """
        提取主声明的值和类型
        返回: (值ID, 值类型)
        """
        if not mainsnak.get('datavalue'):
            return None, None
        
        datavalue = mainsnak['datavalue']
        value_type = datavalue.get('type', '')
        value = datavalue.get('value', '')
        
        if value_type == 'wikibase-entityid':
            return value.get('id', ''), 'entity'
        elif value_type == 'string':
            return value, 'string'
        elif value_type == 'time':
            time_value = value.get('time', '')
            # 清理时间格式：去除+号，只保留日期部分
            if time_value:
                time_value = re.sub(r'^\+(.*)T.*', r'\1', time_value)
            return time_value, 'time'
        elif value_type == 'monolingualtext':
            text = value.get('text', '')
            return text, 'text'
        else:
            value_str = str(value)
            return value_str, 'other'
    
    def extract_time_values(self, time_snaks):
        """
        提取时间值
        """
        time_values = []
        for snak in time_snaks:
            if snak.get('datavalue'):
                datavalue = snak['datavalue']
                if datavalue.get('type') == 'time':
                    time_value = datavalue.get('value', {})
                    # 提取时间字符串（去除+符号和时区信息）
                    time_str = time_value.get('time', '')
                    if time_str:
                        # 清理时间格式：去除+号，只保留日期部分
                        time_str = re.sub(r'^\+(.*)T.*', r'\1', time_str)
                        time_values.append(time_str)
        return time_values
    
    def get_time_status(self, has_start, has_end):
        """
        获取时间状态描述
        """
        if has_start and has_end:
            return "both"
        elif has_start and not has_end:
            return "start_only"
        elif not has_start and has_end:
            return "end_only"
        else:
            return "none"
    
    def process_entities_parallel(self, entity_ids, output_file):
        """
        使用并行处理处理实体列表
        """
        # 将实体ID分成批次
        batches = [entity_ids[i:i+self.batch_size] for i in range(0, len(entity_ids), self.batch_size)]
        
        print(f"将处理 {len(entity_ids)} 个实体，分为 {len(batches)} 批，每批 {self.batch_size} 个")
        print(f"使用 {self.max_workers} 个并行工作线程")
        
        all_temporal_claims = []
        
        # 使用线程池并行处理批次
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            # 提交所有批次任务
            future_to_batch = {executor.submit(self.process_single_batch, batch): batch for batch in batches}
            
            # 收集结果
            for future in tqdm(concurrent.futures.as_completed(future_to_batch), 
                              total=len(batches), desc="处理批次"):
                batch_claims = future.result()
                all_temporal_claims.extend(batch_claims)
        
        # 保存结果到CSV文件
        self.save_results(all_temporal_claims, output_file)
        
        return all_temporal_claims
    
    def process_single_batch(self, batch):
        """
        处理单个批次
        """
        # 批量获取实体数据
        entity_data = self.get_entities_batch(batch)
        
        if not entity_data:
            return []
        
        # 提取时间声明
        temporal_claims = self.extract_temporal_claims({'entities': entity_data})
        
        return temporal_claims
    
    def save_results(self, temporal_claims, output_file):
        """
        保存结果到CSV文件
        """
        if not temporal_claims:
            print("没有找到时间属性声明")
            return
        
        with open(output_file, 'w+', newline='', encoding='utf-8') as f:
            fieldnames = [
                'head_entity_id',      # 头实体ID
                'head_entity_label',   # 头实体标签
                'property_id',         # 属性ID
                'property_label',      # 属性标签
                'tail_entity_id',      # 尾实体ID
                'tail_entity_label',   # 尾实体标签
                'value_type',          # 值类型
                'start_times',         # 开始时间
                'end_times',           # 结束时间
                'time_status'          # 时间状态
            ]
            
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            
            for claim in temporal_claims:
                writer.writerow({
                    'head_entity_id': claim['entity_id'],
                    'head_entity_label': claim['entity_label'],
                    'property_id': claim['property_id'],
                    'property_label': claim['property_label'],
                    'tail_entity_id': claim['main_value_id'],
                    'tail_entity_label': claim['main_value_label'],
                    'value_type': claim['main_value_type'],
                    'start_times': '; '.join(claim['start_times']),
                    'end_times': '; '.join(claim['end_times']),
                    'time_status': claim['time_status']
                })
        
        print(f"结果已保存到: {output_file}")
        print(f"共找到 {len(temporal_claims)} 个时间属性声明")

def main():
    # 示例使用
    entity_ids = [
        "Q76",    # Barack Obama
        "Q90",    # Paris
        "Q114",   # Kenya
        "Q937",   # Albert Einstein
        "Q1339",  # Leonardo da Vinci
        "Q1774"   # Mozart
    ]
    
    output_file = "wikidata_temporal_properties.csv"
    
    # 创建查询器
    query = WikidataTemporalPropertiesOptimized(delay=0.1, max_workers=5)
    
    # 处理实体
    results = query.process_entities_parallel(entity_ids, output_file)
    
    # 打印摘要
    print("\n处理摘要:")
    print(f"处理的实体数量: {len(entity_ids)}")
    print(f"找到的时间属性声明数量: {len(results)}")
    
    # 按时间状态分组
    status_counts = {}
    for claim in results:
        status = claim['time_status']
        status_counts[status] = status_counts.get(status, 0) + 1
    
    print("\n时间状态分布:")
    for status, count in status_counts.items():
        print(f"  {status}: {count}")

def load_entity_ids_from_file(file_path=None):
    """
    从文件加载实体ID列表
    """
    csv_reader = csv.reader(open("results.csv"))
    read_Q_2_original_e = {}

    for row in csv_reader:
        if row[0] == 'original_text':
            continue
        Q_id = row[1]
        original_e = row[0]
        if Q_id != '' and Q_id not in read_Q_2_original_e.keys():
            read_Q_2_original_e[Q_id] = original_e
    print(len(read_Q_2_original_e.keys()))
    
    # 确保是有效的Wikidata ID格式
    entity_ids = []
    for entity in read_Q_2_original_e.keys():
        if entity.startswith('Q') and entity[1:].isdigit():
            entity_ids.append(entity)
        else:
            print(f"跳过无效的实体ID: {entity}")
    print(len(entity_ids))
    return entity_ids

if __name__ == "__main__":
    #main()
    # 从文件加载实体ID
    output_file = "wikidata_temporal_properties.csv"
    
    # 加载实体ID
    entity_ids = load_entity_ids_from_file()
    print(len(entity_ids))
    print(entity_ids[:10])
    
    if not entity_ids:
        print("没有找到有效的实体ID")
        exit()
    
    print(f"加载了 {len(entity_ids)} 个实体ID")
    
    # 创建查询器并处理
    query = WikidataTemporalPropertiesOptimized(delay=0.1, max_workers=10)
    results = query.process_entities_parallel(entity_ids, output_file)

    # 打印摘要
    print("\n处理摘要:")
    print(f"处理的实体数量: {len(entity_ids)}")
    print(f"找到的时间属性声明数量: {len(results)}")
    
    # 按时间状态分组
    status_counts = {}
    for claim in results:
        status = claim['time_status']
        status_counts[status] = status_counts.get(status, 0) + 1
    
    print("\n时间状态分布:")
    for status, count in status_counts.items():
        print(f"  {status}: {count}")
    
    

In [ ]:
# 通过Wikidata_id查询图谱中已有实体的时间区间知识（并行，但返回的是Wikidata_id，需要进一步抽取文本描述和label）
import requests
import json
import csv
import time
import re
import concurrent.futures
from tqdm import tqdm
from collections import defaultdict

class WikidataTemporalPropertiesOptimized:
    def __init__(self, delay=0.1, max_workers=5):
        """
        初始化Wikidata时间属性查询（优化版）
        
        参数:
            delay: 请求间延迟（秒）
            max_workers: 最大并行工作线程数
        """
        self.delay = delay
        self.max_workers = max_workers
        self.headers = {
            'User-Agent': 'WikidataTemporalProperties/2.0 (https://example.com; contact@example.com)'
        }
        
        # 定义时间相关属性
        self.start_time_prop = "P580"  # start time
        self.end_time_prop = "P582"    # end time
        
        # 实体和属性标签缓存
        self.label_cache = {}
        
        # 批量处理相关
        self.batch_size = 50  # 每批处理的实体数量
        
    def get_entities_batch(self, entity_ids, max_retries=3):
        """
        批量获取实体数据
        """
        if not entity_ids:
            return {}
            
        # 将实体ID列表转换为逗号分隔的字符串
        ids_str = "|".join(entity_ids)
        
        url = "https://www.wikidata.org/w/api.php"
        params = {
            'action': 'wbgetentities',
            'ids': ids_str,
            'format': 'json',
            'languages': 'en'
        }
        
        for attempt in range(max_retries):
            try:
                response = requests.get(url, params=params, headers=self.headers, timeout=60)
                
                if response.status_code == 200:
                    data = response.json()
                    return data.get('entities', {})
                elif response.status_code == 429:
                    # 处理速率限制
                    wait_time = 10 * (attempt + 1)
                    print(f"速率限制，等待{wait_time}秒后重试")
                    time.sleep(wait_time)
                    continue
                elif response.status_code == 404:
                    print(f"部分实体不存在")
                    return {}
                else:
                    print(f"HTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
            except requests.exceptions.RequestException as e:
                print(f"请求异常: {e}")
                if attempt < max_retries - 1:
                    time.sleep(5)
        
        return {}
    
    def get_labels_batch(self, entity_ids, max_retries=10):
        """
        批量获取实体标签
        """
        if not entity_ids:
            return {}
            
        # 过滤掉非实体ID
        q_ids = [eid for eid in entity_ids if eid.startswith('Q') or eid.startswith('P')]
        if not q_ids:
            return {}
            
        # 将实体ID列表转换为逗号分隔的字符串
        ids_str = "|".join(q_ids)
        
        url = "https://www.wikidata.org/w/api.php"
        params = {
            'action': 'wbgetentities',
            'ids': ids_str,
            'format': 'json',
            'languages': 'en',
            'props': 'labels'
        }
        
        for attempt in range(max_retries):
                response = requests.get(url, params=params, headers=self.headers, timeout=60)
                print(response)
                if response.status_code == 200:
                    data = response.json()
                    entities = data.get('entities', {})
                    
                    labels = {}
                    for eid, entity in entities.items():
                        labels_data = entity.get('labels', {})
                        if 'en' in labels_data:
                            labels[eid] = labels_data['en'].get('value', eid)
                        else:
                            # 如果没有英文标签，使用第一个可用标签
                            for lang in labels_data:
                                if labels_data[lang].get('value'):
                                    labels[eid] = labels_data[lang].get('value')
                                    break
                            else:
                                labels[eid] = eid
                    
                    # 更新缓存
                    self.label_cache.update(labels)
                    return labels
                elif response.status_code == 429:
                    # 处理速率限制
                    wait_time = 10 * (attempt + 1)
                    print(f"速率限制，等待{wait_time}秒后重试")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"获取标签时HTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
                '''
                except requests.exceptions.RequestException as e:
                    print(f"获取标签时请求异常: {e}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
                '''
        
        return {}
    
    def extract_temporal_claims(self, entity_data):
        """
        从实体数据中提取包含时间信息的声明
        """
        temporal_claims = []
        
        if not entity_data or 'entities' not in entity_data:
            return temporal_claims
        
        # 收集所有需要获取标签的实体ID
        entities_to_label = set()
        
        for entity_id, entity in entity_data['entities'].items():
            # 添加头实体ID到标签查询列表
            entities_to_label.add(entity_id)
            
            if 'claims' not in entity:
                continue
                
            # 遍历所有属性
            for prop_id, claims in entity['claims'].items():
                entities_to_label.add(prop_id)  # 添加属性ID
                
                for claim in claims:
                    # 检查声明是否有限定符
                    if 'qualifiers' in claim:
                        qualifiers = claim['qualifiers']
                        
                        # 检查是否有start time或end time限定符
                        has_start_time = self.start_time_prop in qualifiers
                        has_end_time = self.end_time_prop in qualifiers
                        
                        if has_start_time or has_end_time:
                            # 获取主声明值
                            mainsnak = claim.get('mainsnak', {})
                            datatype = mainsnak.get('datatype', '')
                            datavalue = mainsnak.get('datavalue', {})
                            
                            # 跳过数字类型的值
                            if datatype == 'quantity':
                                continue
                            
                            # 提取主声明值
                            main_value_id, main_value_type = self.extract_main_value_info(mainsnak)
                            if main_value_id and main_value_id.startswith(('Q', 'P')):
                                entities_to_label.add(main_value_id)
                            
                            # 提取时间限定符
                            start_times = self.extract_time_values(qualifiers.get(self.start_time_prop, []))
                            end_times = self.extract_time_values(qualifiers.get(self.end_time_prop, []))
                            
                            temporal_claims.append({
                                'entity_id': entity_id,
                                'property_id': prop_id,
                                'main_value_id': main_value_id,
                                'main_value_type': main_value_type,
                                'start_times': start_times,
                                'end_times': end_times,
                                'has_start_time': has_start_time,
                                'has_end_time': has_end_time,
                                'time_status': self.get_time_status(has_start_time, has_end_time)
                            })
        
        # 批量获取所有需要的标签
        self.get_labels_batch(list(entities_to_label))
        
        # 为每个声明添加标签
        for claim in temporal_claims:
            # 头实体标签
            claim['entity_label'] = self.label_cache.get(claim['entity_id'], claim['entity_id'])
            
            # 属性标签
            claim['property_label'] = self.label_cache.get(claim['property_id'], claim['property_id'])
            
            # 尾实体标签
            main_value_id = claim['main_value_id']
            if main_value_id and main_value_id.startswith(('Q', 'P')):
                claim['main_value_label'] = self.label_cache.get(main_value_id, main_value_id)
            else:
                claim['main_value_label'] = main_value_id
        
        return temporal_claims
    
    def extract_main_value_info(self, mainsnak):
        """
        提取主声明的值和类型
        返回: (值ID, 值类型)
        """
        if not mainsnak.get('datavalue'):
            return None, None
        
        datavalue = mainsnak['datavalue']
        value_type = datavalue.get('type', '')
        value = datavalue.get('value', '')
        
        if value_type == 'wikibase-entityid':
            return value.get('id', ''), 'entity'
        elif value_type == 'string':
            return value, 'string'
        elif value_type == 'time':
            time_value = value.get('time', '')
            # 清理时间格式：去除+号，只保留日期部分
            if time_value:
                time_value = re.sub(r'^\+(.*)T.*', r'\1', time_value)
            return time_value, 'time'
        elif value_type == 'monolingualtext':
            text = value.get('text', '')
            return text, 'text'
        else:
            value_str = str(value)
            return value_str, 'other'
    
    def extract_time_values(self, time_snaks):
        """
        提取时间值
        """
        time_values = []
        for snak in time_snaks:
            if snak.get('datavalue'):
                datavalue = snak['datavalue']
                if datavalue.get('type') == 'time':
                    time_value = datavalue.get('value', {})
                    # 提取时间字符串（去除+符号和时区信息）
                    time_str = time_value.get('time', '')
                    if time_str:
                        # 清理时间格式：去除+号，只保留日期部分
                        time_str = re.sub(r'^\+(.*)T.*', r'\1', time_str)
                        time_values.append(time_str)
        return time_values
    
    def get_time_status(self, has_start, has_end):
        """
        获取时间状态描述
        """
        if has_start and has_end:
            return "both"
        elif has_start and not has_end:
            return "start_only"
        elif not has_start and has_end:
            return "end_only"
        else:
            return "none"
    
    def process_entities_parallel(self, entity_ids, output_file):
        """
        使用并行处理处理实体列表
        """
        # 将实体ID分成批次
        batches = [entity_ids[i:i+self.batch_size] for i in range(0, len(entity_ids), self.batch_size)]
        
        print(f"将处理 {len(entity_ids)} 个实体，分为 {len(batches)} 批，每批 {self.batch_size} 个")
        print(f"使用 {self.max_workers} 个并行工作线程")
        
        all_temporal_claims = []
        
        # 使用线程池并行处理批次
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            # 提交所有批次任务
            future_to_batch = {executor.submit(self.process_single_batch, batch): batch for batch in batches}
            
            # 收集结果
            for future in tqdm(concurrent.futures.as_completed(future_to_batch), 
                              total=len(batches), desc="处理批次"):
                batch_claims = future.result()
                all_temporal_claims.extend(batch_claims)
        
        # 保存结果到CSV文件
        self.save_results(all_temporal_claims, output_file)
        
        return all_temporal_claims
    
    def process_single_batch(self, batch):
        """
        处理单个批次
        """
        # 批量获取实体数据
        entity_data = self.get_entities_batch(batch)
        
        if not entity_data:
            return []
        
        # 提取时间声明
        temporal_claims = self.extract_temporal_claims({'entities': entity_data})
        
        return temporal_claims
    
    def save_results(self, temporal_claims, output_file):
        """
        保存结果到CSV文件
        """
        if not temporal_claims:
            print("没有找到时间属性声明")
            return
        
        with open(output_file, 'w+', newline='', encoding='utf-8') as f:
            fieldnames = [
                'head_entity_id',      # 头实体ID
                'head_entity_label',   # 头实体标签
                'property_id',         # 属性ID
                'property_label',      # 属性标签
                'tail_entity_id',      # 尾实体ID
                'tail_entity_label',   # 尾实体标签
                'value_type',          # 值类型
                'start_times',         # 开始时间
                'end_times',           # 结束时间
                'time_status'          # 时间状态
            ]
            
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            
            for claim in temporal_claims:
                writer.writerow({
                    'head_entity_id': claim['entity_id'],
                    'head_entity_label': claim['entity_label'],
                    'property_id': claim['property_id'],
                    'property_label': claim['property_label'],
                    'tail_entity_id': claim['main_value_id'],
                    'tail_entity_label': claim['main_value_label'],
                    'value_type': claim['main_value_type'],
                    'start_times': '; '.join(claim['start_times']),
                    'end_times': '; '.join(claim['end_times']),
                    'time_status': claim['time_status']
                })
        
        print(f"结果已保存到: {output_file}")
        print(f"共找到 {len(temporal_claims)} 个时间属性声明")

def load_entity_ids_from_file(file_path=None):
    """
    从文件加载实体ID列表
    """
    csv_reader = csv.reader(open("wikidata_temporal_properties.csv"))
    read_Q_2_original_e = {}

    for row in csv_reader:
        if row[0] == 'head_entity_id':
            continue
        Q_id = row[4]
        original_e = row[5]
        if Q_id != '' and Q_id not in read_Q_2_original_e.keys():
            read_Q_2_original_e[Q_id] = original_e
    print(len(read_Q_2_original_e.keys()))
    
    # 确保是有效的Wikidata ID格式
    entity_ids = []
    for entity in read_Q_2_original_e.keys():
        if entity.startswith('Q') and entity[1:].isdigit():
            entity_ids.append(entity)
        else:
            print(f"跳过无效的实体ID: {entity}")
    print(len(entity_ids))
    return entity_ids

if __name__ == "__main__":
    #main()
    # 从文件加载实体ID
    output_file = "wikidata_temporal_properties_hop2.csv"
    
    # 加载实体ID
    entity_ids = load_entity_ids_from_file()
    print(len(entity_ids))
    print(entity_ids[:10])
    
    if not entity_ids:
        print("没有找到有效的实体ID")
        exit()
    
    print(f"加载了 {len(entity_ids)} 个实体ID")
    
    # 创建查询器并处理
    query = WikidataTemporalPropertiesOptimized(delay=0.1, max_workers=10)
    results = query.process_entities_parallel(entity_ids, output_file)

    # 打印摘要
    print("\n处理摘要:")
    print(f"处理的实体数量: {len(entity_ids)}")
    print(f"找到的时间属性声明数量: {len(results)}")
    
    # 按时间状态分组
    status_counts = {}
    for claim in results:
        status = claim['time_status']
        status_counts[status] = status_counts.get(status, 0) + 1
    
    print("\n时间状态分布:")
    for status, count in status_counts.items():
        print(f"  {status}: {count}")
    
    

In [ ]:
import csv

remain_entities = set()
read_entites = set()

csv_reader = csv.reader(open("wikidata_temporal_properties.csv"))
for row in csv_reader:
    if row[0] == 'head_entity_id':
        continue
    s_Q_id = row[0]
    o_Q_id = row[4]
    remain_entities.add(s_Q_id)
    remain_entities.add(o_Q_id)

csv_reader = csv.reader(open("wikidata_temporal_properties_hop2.csv"))
for row in csv_reader:
    if row[0] == 'head_entity_id':
        continue
    s_Q_id = row[0]
    o_Q_id = row[4]
    remain_entities.add(s_Q_id)
    remain_entities.add(o_Q_id)

csv_reader = csv.reader(open("results.csv"))

for row in csv_reader:
    if row[0] == 'original_text':
        continue
    Q_id = row[1]
    if Q_id != '' and Q_id not in read_entites:
        read_entites.add(Q_id)

print(len(remain_entities))

In [ ]:
'Q509404' in remain_entities

In [ ]:
# 抓取新实体的label 和 short description
import requests
import csv
import time
import json
from tqdm import tqdm
import re
import concurrent.futures

class WikidataLabelDescriptionFetcher:
    def __init__(self, delay=0.1, max_workers=10, batch_size=50):
        """
        初始化Wikidata标签和描述抓取器
        
        参数:
            delay: 请求间延迟（秒）
            max_workers: 最大并行工作线程数
            batch_size: 每批处理的实体数量
        """
        self.delay = delay
        self.max_workers = max_workers
        self.batch_size = batch_size
        self.headers = {
            'User-Agent': 'WikidataLabelDescriptionFetcher/1.0 (https://example.com; contact@example.com)'
        }
        
        # 结果缓存，避免重复查询
        self.result_cache = {}
    
    def fetch_entities_batch(self, entity_ids, max_retries=3):
        """
        批量获取实体标签和描述
        """
        if not entity_ids:
            return {}
            
        # 过滤已缓存的实体
        uncached_ids = [eid for eid in entity_ids if eid not in self.result_cache]
        
        if not uncached_ids:
            # 所有实体都已缓存，直接返回
            return {eid: self.result_cache[eid] for eid in entity_ids}
        
        # 将实体ID列表转换为逗号分隔的字符串
        ids_str = "|".join(uncached_ids)
        
        url = "https://www.wikidata.org/w/api.php"
        params = {
            'action': 'wbgetentities',
            'ids': ids_str,
            'format': 'json',
            'languages': 'en',
            'props': 'labels|descriptions'
        }
        
        for attempt in range(max_retries):
            try:
                response = requests.get(url, params=params, headers=self.headers, timeout=30)
                
                if response.status_code == 200:
                    data = response.json()
                    entities = data.get('entities', {})
                    
                    results = {}
                    for eid in entity_ids:
                        if eid in self.result_cache:
                            # 使用缓存的结果
                            results[eid] = self.result_cache[eid]
                        elif eid in entities:
                            # 提取标签和描述
                            entity = entities[eid]
                            
                            # 获取标签
                            labels = entity.get('labels', {})
                            label = labels.get('en', {}).get('value', '') if 'en' in labels else ''
                            
                            # 获取描述
                            descriptions = entity.get('descriptions', {})
                            description = descriptions.get('en', {}).get('value', '') if 'en' in descriptions else ''
                            
                            # 清理描述中的时间信息
                            cleaned_description = self.clean_description(description)
                            
                            result = {
                                'label': label,
                                'description': cleaned_description,
                                'original_description': description
                            }
                            
                            results[eid] = result
                            # 更新缓存
                            self.result_cache[eid] = result
                        else:
                            # 实体不存在或无法获取
                            results[eid] = {
                                'label': '',
                                'description': '',
                                'original_description': ''
                            }
                    
                    return results
                elif response.status_code == 429:
                    # 处理速率限制
                    wait_time = 10 * (attempt + 1)
                    print(f"速率限制，等待{wait_time}秒后重试")
                    time.sleep(wait_time)
                    continue
                elif response.status_code == 404:
                    print(f"部分实体不存在")
                    # 返回空结果
                    return {eid: {'label': '', 'description': '', 'original_description': ''} for eid in entity_ids}
                else:
                    print(f"HTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
            except requests.exceptions.RequestException as e:
                print(f"请求异常: {e}")
                if attempt < max_retries - 1:
                    time.sleep(5)
        
        # 如果所有重试都失败，返回空结果
        return {eid: {'label': '', 'description': '', 'original_description': ''} for eid in entity_ids}
    
    def clean_description(self, description):
        """
        清理描述文本，移除时间相关的内容
        """
        if not description:
            return ""
        
        # 时间模式正则表达式
        time_patterns = [
            r'\(\s*\d{4}\s*[-–—]\s*\d{4}\s*\)',  # (1929-2003)
            r'\(\s*\d{4}\s*\)',  # (1929)
            r'from\s+\d{4}\s+to\s+\d{4}',  # from 1929 to 2003
            r'since\s+\d{4}\s+to\s+\d{4}',  # since 1929 to 2003
            r'\d{4}\s*[-–—]\s*\d{4}',  # 1929-2003 (无括号)
            r'born\s+\d{4}',  # born 1929
            r'\b\d{4}\s*-\s*\d{4}\b',  # 1929 - 2003
            r'\b\d{4}\s*–\s*\d{4}\b',  # 1929 – 2003 (不同种类的横线)
            r'\b\d{4}\s*—\s*\d{4}\b',  # 1929 — 2003
        ]
        
        cleaned = description
        
        # 应用所有时间模式
        for pattern in time_patterns:
            cleaned = re.sub(pattern, '', cleaned, flags=re.IGNORECASE)
        
        # 清理多余的空格和标点
        cleaned = re.sub(r'\s+', ' ', cleaned)  # 多个空格变一个
        cleaned = re.sub(r'\s*,\s*,', ',', cleaned)  # 清理多余的逗号
        cleaned = re.sub(r'^\s*[,-]\s*', '', cleaned)  # 开头是逗号或横线
        cleaned = re.sub(r'\s*[,-]\s*$', '', cleaned)  # 结尾是逗号或横线
        cleaned = cleaned.strip()
        
        return cleaned
    
    def process_entities_parallel(self, entity_ids, output_file):
        """
        使用并行处理处理实体列表
        """
        # 将实体ID分成批次
        batches = [entity_ids[i:i+self.batch_size] for i in range(0, len(entity_ids), self.batch_size)]
        
        print(f"将处理 {len(entity_ids)} 个实体，分为 {len(batches)} 批，每批 {self.batch_size} 个")
        print(f"使用 {self.max_workers} 个并行工作线程")
        
        all_results = {}
        
        # 使用线程池并行处理批次
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            # 提交所有批次任务
            future_to_batch = {executor.submit(self.fetch_entities_batch, batch): batch for batch in batches}
            
            # 收集结果
            for future in tqdm(concurrent.futures.as_completed(future_to_batch), 
                              total=len(batches), desc="处理批次"):
                batch_results = future.result()
                all_results.update(batch_results)
        
        # 保存结果到CSV文件
        self.save_results(all_results, output_file)
        
        return all_results
    
    def save_results(self, results, output_file):
        """
        保存结果到CSV文件
        """
        if not results:
            print("没有找到任何结果")
            return
        
        with open(output_file, 'a', newline='', encoding='utf-8') as f:
            fieldnames = [
                'entity_id',
                'label',
                'description',
                'original_description'
            ]
            
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            
            for entity_id, data in results.items():
                writer.writerow({
                    'entity_id': entity_id,
                    'label': data['label'],
                    'description': data['description'],
                    'original_description': data['original_description']
                })
        
        print(f"结果已保存到: {output_file}")
        print(f"共处理 {len(results)} 个实体")

def main():
    # 示例使用
    entity_ids = [
        "Q76",    # Barack Obama
        "Q90",    # Paris
        "Q114",   # Kenya
        "Q937",   # Albert Einstein
        "Q1339",  # Leonardo da Vinci
        "Q1774"   # Mozart
    ]
    
    output_file = "wikidata_labels_descriptions.csv"
    
    # 创建抓取器
    fetcher = WikidataLabelDescriptionFetcher(delay=0.1, max_workers=10, batch_size=50)
    
    # 处理实体
    results = fetcher.process_entities_parallel(entity_ids, output_file)
    
    # 打印摘要
    print("\n处理摘要:")
    print(f"处理的实体数量: {len(entity_ids)}")
    
    # 统计有标签和描述的实体数量
    with_label = sum(1 for data in results.values() if data['label'])
    with_description = sum(1 for data in results.values() if data['description'])
    
    print(f"有标签的实体数量: {with_label}")
    print(f"有描述的实体数量: {with_description}")

def load_entity_ids_from_file(file):
    """
    从文件加载实体ID列表
    """
    # 确保是有效的Wikidata ID格式
    entity_ids = []
    for entity in file:
        if entity.startswith('Q') and entity[1:].isdigit():
            entity_ids.append(entity)
        else:
            print(f"跳过无效的实体ID: {entity}")
    
    return entity_ids

if __name__ == "__main__":
    
    # 从文件加载实体ID
    output_file = "new_entity_labels_descriptions.csv"
    
    entity_ids = load_entity_ids_from_file(remain_entities)
    
    if not entity_ids:
        print("没有找到有效的实体ID")
        exit()
    
    print(f"加载了 {len(entity_ids)} 个实体ID")
    
    # 创建抓取器并处理
    fetcher = WikidataLabelDescriptionFetcher(delay=0.1, max_workers=10, batch_size=50)
    results = fetcher.process_entities_parallel(entity_ids, output_file)

In [ ]:
# 筛选有wikidata_id 的实体对应知识
import csv
import os
filePath = ''
file_list = os.listdir(filePath)
print(file_list)

csv_reader = csv.reader(open("results.csv"))
original_e_2_read_Q_label_des = {}
stamp_Q_set = set()

for row in csv_reader:
    if row[0] == 'original_text':
        continue
    Q_id = row[1]
    original_e = row[0]
    read_label = row[2]
    read_des = row[3]
    if Q_id != '' and original_e not in original_e_2_read_Q_label_des.keys():
        original_e_2_read_Q_label_des[original_e] = [Q_id, read_label, read_des]
        stamp_Q_set.add(Q_id)
print(len(original_e_2_read_Q_label_des.keys()))

import pandas as pd
import tqdm
import csv
ICEWS_all_entites = set()
ICEWS_all_stamp_fact = set()

e_2_des = {}

for file_name in tqdm.tqdm(file_list):
    if '.tab' in file_name:
        with open(file_name, 'r') as f:
            text = f.read()
        data = text.split('\n')
        for row in data:
            sperate_row = row.strip().split('\t')
            if len(sperate_row) != 20:
                continue
            time = sperate_row[1]
            if time == 'Event Date':
                continue
            s = sperate_row[2]
            s_des = 'Sectors: ' + sperate_row[3] + ' Country:' + sperate_row[4]
            r = sperate_row[5]
            o = sperate_row[8]
            o_des = 'Sectors: ' + sperate_row[9] + ' Country:' + sperate_row[10]
            ICEWS_all_stamp_fact.add((s, r, o, time))
            ICEWS_all_entites.add(s)
            ICEWS_all_entites.add(o)

            if s not in e_2_des.keys():
                e_2_des[s] = s_des
            if o not in e_2_des.keys():
                e_2_des[o] = o_des
    
    
    elif '.tsv' in file_name:
        with open(file_name, 'r') as f:
            text = f.read()
        data = text.split('\n')
    
        for row in data:
            sperate_row = row.strip().split('\t')
            if len(sperate_row) != 20:
                continue
            time = sperate_row[1]
            if time == 'Event Date':
                continue
            s = sperate_row[2]
            s_des = 'Sectors: ' + sperate_row[3] + ' Country:' + sperate_row[4]
            r = sperate_row[5]
            o = sperate_row[8]
            o_des = 'Sectors: ' + sperate_row[9] + ' Country:' + sperate_row[10]
            ICEWS_all_stamp_fact.add((s, r, o, time))
            ICEWS_all_entites.add(s)
            ICEWS_all_entites.add(o)

            if s not in e_2_des.keys():
                e_2_des[s] = s_des
            if o not in e_2_des.keys():
                e_2_des[o] = o_des

print(len(ICEWS_all_entites))
print(len(ICEWS_all_stamp_fact))

#print(e_2_des)
print(list(ICEWS_all_entites)[:10])
print(list(ICEWS_all_entites)[:10])
print([(key, e_2_des[key]) for key in list(e_2_des.keys())[:30]])


wiki_Q_stamp_facts = set()
for f in ICEWS_all_stamp_fact:
    if f[0] not in original_e_2_read_Q_label_des.keys() or f[2] not in original_e_2_read_Q_label_des.keys():
        continue
    wiki_Q_stamp_facts.add(f)
print(len(wiki_Q_stamp_facts))


# 将包含 Wikidata id 的实体的描述改为label

final_e_2_keywords_dict = {}
stamp_knowledge = set()
min_time = '2025-10-31'
max_time = '0000-00-00'
for f in ICEWS_all_stamp_fact:
    if f[0] not in original_e_2_read_Q_label_des.keys() or f[2] not in original_e_2_read_Q_label_des.keys():
        continue
    stamp_knowledge.add((original_e_2_read_Q_label_des[f[0]][1], f[1], original_e_2_read_Q_label_des[f[2]][1], f[3]))
    if original_e_2_read_Q_label_des[f[0]][1] not in final_e_2_keywords_dict.keys():
        final_e_2_keywords_dict[original_e_2_read_Q_label_des[f[0]][1]] = original_e_2_read_Q_label_des[f[0]][2]
    if original_e_2_read_Q_label_des[f[2]][1] not in final_e_2_keywords_dict.keys():
        final_e_2_keywords_dict[original_e_2_read_Q_label_des[f[2]][1]] = original_e_2_read_Q_label_des[f[2]][2]
    
    if f[3] < min_time:
        min_time = f[3]
    if f[3] > max_time:
        max_time = f[3]

print(len(stamp_knowledge))
print(len(final_e_2_keywords_dict.keys()))
print(list(stamp_knowledge)[:10])
print([min_time, max_time])


In [ ]:
import tqdm

# 时间区间知识
id_2_rel = {}
final_span_knowlegde = set()
csv_reader = csv.reader(open("new_property_labels.csv"))
for row in csv_reader:
	if row[0] == 'Property_ID':
		continue
	id_2_rel[row[0]] = row[1]

#读取时间知识相关实体的id-label-des映射
csv_reader = csv.reader(open("new_entity_labels_descriptions.csv"))
Q_2_label_des = {}

for row in csv_reader:
    if row[0] == 'entity_id':
        continue
    Q_id = row[0]
    read_label = row[1]
    read_des = row[2]
    if read_label != '' and Q_id not in Q_2_label_des.keys():
        Q_2_label_des[Q_id] = [read_label, read_des]
    if read_label != '' and read_label not in final_e_2_keywords_dict.keys():
        final_e_2_keywords_dict[read_label] = read_des

print(len(Q_2_label_des.keys()))

# 加入时间区间知识
span_knowledge = set()
span_e_set = set()

csv_reader = csv.reader(open("wikidata_temporal_properties.csv"))
for row in tqdm.tqdm(csv_reader):
    if row[0] == 'head_entity_id':
        continue
    if row[6] != 'entity':
        continue
    if row[0] not in Q_2_label_des.keys() or row[4] not in Q_2_label_des.keys():
        continue

    s_Q_id = Q_2_label_des[row[0]][0]
    r_id = id_2_rel[row[2]]
    o_Q_id = Q_2_label_des[row[4]][0]
    start_time = row[7].split(';')[0]
    end_time = row[8].split(';')[0]

    if start_time == '' and end_time > min_time:
        start_time = min_time
        if end_time > max_time:
            end_time = max_time
    if end_time == '' and start_time < max_time:
        end_time = max_time
        if start_time < min_time:
            start_time = min_time
    
    if end_time < min_time or start_time > max_time:
        continue
    if start_time < min_time:
        start_time = min_time
    if end_time > max_time:
        end_time = max_time
    
    if start_time == '' and end_time == '':
        continue
    if start_time > end_time:
        continue

    span_knowledge.add((s_Q_id, r_id, o_Q_id, start_time, end_time))
    span_e_set.add(row[0])
    span_e_set.add(row[4])

csv_reader = csv.reader(open("wikidata_temporal_properties_hop2.csv"))
for row in tqdm.tqdm(csv_reader):
    if row[0] == 'head_entity_id':
        continue
    if row[6] != 'entity':
        continue
    if row[0] not in Q_2_label_des.keys() or row[4] not in Q_2_label_des.keys():
        continue
    s_Q_id = Q_2_label_des[row[0]][0]
    r_id = id_2_rel[row[2]]
    o_Q_id = Q_2_label_des[row[4]][0]
    start_time = row[7].split(';')[0]
    end_time = row[8].split(';')[0]

    if start_time == '' and end_time > min_time:
        start_time = min_time
        if end_time > max_time:
            end_time = max_time
    if end_time == '' and start_time < max_time:
        end_time = max_time
        if start_time < min_time:
            start_time = min_time
    
    if end_time < min_time or start_time > max_time:
        continue
    if start_time < min_time:
        start_time = min_time
    if end_time > max_time:
        end_time = max_time    
    
    if start_time == '' and end_time == '':
        continue
    if start_time > end_time:
        continue

    span_knowledge.add((s_Q_id, r_id, o_Q_id, start_time, end_time))
    span_e_set.add(row[0])
    span_e_set.add(row[4])

print(len(span_e_set))
print(len(stamp_Q_set & span_e_set))
print(len(set([original_e_2_read_Q_label_des[e][1] for e in ICEWS_all_entites if e in original_e_2_read_Q_label_des.keys()]) & set([Q_2_label_des[e][0] for e in span_e_set if e in Q_2_label_des.keys()])))
print(len(span_knowledge))
print(list(span_knowledge)[:10])
print(list(stamp_Q_set)[:10])
print(list(span_e_set)[:10])

In [ ]:
# check
#check
all_knowledge = []

for f in stamp_knowledge:
    #print(f)
    if f[0] not in final_e_2_keywords_dict.keys() or f[2] not in final_e_2_keywords_dict.keys():
        print("entity error: "+str(f))
    all_knowledge.append(f)

for f in span_knowledge:
    if f[0] not in final_e_2_keywords_dict.keys() or f[2] not in final_e_2_keywords_dict.keys():
        print("entity error: "+str(f))
    if f[-2] > f[-1]:
        print("time error: "+str(f))
        continue
    all_knowledge.append(f)

print(len(all_knowledge))
print(len(final_e_2_keywords_dict.keys()))

In [ ]:
import pickle
pickle.dump([final_e_2_keywords_dict, all_knowledge], open('ICEWSWiki_e60k_f5700k.pkl', 'wb'))

In [ ]:
node_2_keys, fact = pickle.load(open('ICEWSWiki_e60k_f5700k.pkl', 'rb'))
print([(key, node_2_keys[key]) for key in list(node_2_keys.keys())[:30]])
print(list(fact)[:30])

In [ ]:
import pickle
node_2_keys, fact = pickle.load(open('ICEWSWiki_e60k_f5700k.pkl', 'rb'))
print([(key, node_2_keys[key]) for key in list(node_2_keys.keys())[:30]])
print(list(fact)[:30])

In [ ]:
import matplotlib.pyplot as plt
import tqdm
from datetime import date, timedelta

def get_dates_between_simple(start_date, end_date):
    """
    简化版的日期遍历函数
    """
    try:
        start = date.fromisoformat(start_date)
        end = date.fromisoformat(end_date)
    except:
        start = date.fromisoformat(start_date)
        end_date = end_date[:-1] + '0'
        end = date.fromisoformat(end_date)
    
    dates = []
    current = start
    while current <= end:
        dates.append(current.isoformat())
        current += timedelta(days=1)
    
    return dates

# split train valid test
# 首先确定切分时间戳
all_times = set()
t_2_stamp_fact_num = {}
t_2_span_fact_num = {}
t_2_start_num = {}
t_2_end_num = {}
new_fact = set()
for f in tqdm.tqdm(fact):
    if len(f) == 4:
        if '-00' in f[-1]:
            continue
        all_times.add(f[-1])
        if f[-1] not in t_2_stamp_fact_num.keys():
            t_2_stamp_fact_num[f[-1]] = 0
        t_2_stamp_fact_num[f[-1]] += 1 
        new_fact.add(f)
    if len(f) == 5:
        if '-00' in f[-1] or '-00' in f[-2]:
            continue
        if f[-1] <= f[-2]:
            continue
        all_times.add(f[-1])
        all_times.add(f[-2])
        for t in get_dates_between_simple(f[-2], f[-1]): #range(f[-2], f[-1]+1):
            if t not in t_2_span_fact_num.keys():
                t_2_span_fact_num[t] = 0
            t_2_span_fact_num[t] += 1 
        
        if f[-2] not in t_2_start_num.keys():
            t_2_start_num[f[-2]] = 0
        t_2_start_num[f[-2]] += 1

        if f[-1] not in t_2_end_num.keys():
            t_2_end_num[f[-1]] = 0
        t_2_end_num[f[-1]] += 1

        if f[-1][-2:] == '31' and t[-2:] == '30':
            if f[3] < t:
                new_fact.add((f[0], f[1], f[2], f[3], t))
        else:
            new_fact.add(f)


sorted_all_times = sorted(list(all_times))
sorted_stamp_nums = []
sorted_span_nums = []
sorted_start_nums = []
sorted_end_nums = []
for t in sorted_all_times:
    if t in t_2_stamp_fact_num.keys():
        sorted_stamp_nums.append(t_2_stamp_fact_num[t])
    else:
        sorted_stamp_nums.append(0)
    
    if t in t_2_span_fact_num.keys():
        sorted_span_nums.append(t_2_span_fact_num[t])
    else:
        sorted_span_nums.append(0)

    if t in t_2_start_num.keys():
        sorted_start_nums.append(t_2_start_num[t])
    else:
        sorted_start_nums.append(0)
    
    if t in t_2_end_num.keys():
        sorted_end_nums.append(t_2_end_num[t])
    else:
        sorted_end_nums.append(0)
print(sorted_all_times)
print(len(sorted_all_times))
print(sorted_stamp_nums)
print(len(sorted_stamp_nums))
print(sorted_span_nums)
print(len(sorted_span_nums))


In [ ]:
split_ratio = 0.6 # 0.9 for e40k_f130k
valid_preiod = 0.025 # 0.025 for e40k_f130k
# 切分时间戳
print('train/valid/test split point')
print([int(split_ratio*len(sorted_all_times)), sorted_all_times[int(split_ratio*len(sorted_all_times))]])
print([int((split_ratio+valid_preiod)*len(sorted_all_times)), sorted_all_times[int((split_ratio+valid_preiod)*len(sorted_all_times))]])

print('train stamp span fact num')
# 切分后的训练集时间戳知识数量
print(sum(sorted_stamp_nums[:int(split_ratio*len(sorted_all_times))]))
# 切分后的训练集时间区间知识数量
print(sum(sorted_span_nums[:int(split_ratio*len(sorted_all_times))]))

print('valid stamp span fact num')
# 切分后的验证集时间戳知识数量
print(sum(sorted_stamp_nums[int(split_ratio*len(sorted_all_times)):int((split_ratio+valid_preiod)*len(sorted_all_times))]))
# 切分后的验证集时间区间知识数量
print(sum(sorted_span_nums[int(split_ratio*len(sorted_all_times)):int((split_ratio+valid_preiod)*len(sorted_all_times))]))

print('test stamp span fact num')
# 切分后的测试集时间戳知识数量
print(sum(sorted_stamp_nums[int((split_ratio+valid_preiod)*len(sorted_all_times)):]))
# 切分后的测试集时间区间知识数量
print(sum(sorted_span_nums[int((split_ratio+valid_preiod)*len(sorted_all_times)):]))

# 检查验证集和测试集各有多少时间区间知识结束和时间区间知识开始
print('train start end fact num')
print(sum(sorted_start_nums[:int(split_ratio*len(sorted_all_times))]))
print(sum(sorted_end_nums[:int(split_ratio*len(sorted_all_times))]))
print('valid start end fact num')
print(sum(sorted_start_nums[int(split_ratio*len(sorted_all_times)):int((split_ratio+valid_preiod)*len(sorted_all_times))]))
print(sum(sorted_end_nums[int(split_ratio*len(sorted_all_times)):int((split_ratio+valid_preiod)*len(sorted_all_times))]))

print('test start end fact num')
print(sum(sorted_start_nums[int((split_ratio+valid_preiod)*len(sorted_all_times)):-1]))
print(sum(sorted_end_nums[int((split_ratio+valid_preiod)*len(sorted_all_times)):-1]))

plt.plot(sorted_stamp_nums, color='skyblue', alpha=0.5)
plt.plot(sorted_span_nums, color='red', alpha=0.5)
plt.show()

In [ ]:
# 正式切分数据集
train_set = set()
valid_set = set()
test_set = set()

train_split_time = sorted_all_times[int(split_ratio*len(sorted_all_times))-1]
valid_split_time = sorted_all_times[int((split_ratio+valid_preiod)*len(sorted_all_times))-1]
for f in new_fact:
    if len(f) == 4: # stamp knowledge
        s, r, o, t = f
        if t <= train_split_time:
            train_set.add(f)
        elif t <= valid_split_time:
            valid_set.add(f)
        else:
            test_set.add(f)
    if len(f) == 5: # span knowledge
        s, r, o, t_s, t_o = f
        if t_s <= train_split_time:
            train_set.add(f)
        elif t_s <= valid_split_time:
            valid_set.add(f)
        else:
            test_set.add(f)
        
        if t_o > train_split_time:
            valid_set.add(f)
        if t_o > valid_split_time:
            test_set.add(f)

print([len(train_set), len(valid_set), len(test_set)])
print(len(train_set|valid_set|test_set))

# check the consistency
print('train stamp span fact num')
# 切分后的训练集时间戳知识数量
print(len([f for f in train_set if len(f) == 4]))
# 切分后的训练集时间区间知识数量
print(len([f for f in train_set if len(f) == 5]))

print('valid stamp span fact num')
# 切分后的训练集时间戳知识数量
print(len([f for f in valid_set if len(f) == 4]))
# 切分后的训练集时间区间知识数量
print(len([f for f in valid_set if len(f) == 5]))

print('test stamp span fact num')
# 切分后的训练集时间戳知识数量
print(len([f for f in test_set if len(f) == 4]))
# 切分后的训练集时间区间知识数量
print(len([f for f in test_set if len(f) == 5]))

# 检查验证集和测试集各有多少时间区间知识结束和时间区间知识开始
print('train start end fact num')
start_num, end_num = 0, 0
train_start_fact = set()
train_end_fact = set()
for f in train_set:
    if len(f) == 5:
        if f[-2] <= train_split_time:
            start_num += 1
            train_start_fact.add(f)
        if f[-1] <= train_split_time:
            end_num += 1
            train_end_fact.add(f)
print([start_num, end_num])

print('valid start end fact num')
start_num, end_num = 0, 0
valid_start_fact = set()
valid_end_fact = set()
for f in valid_set:
    if len(f) == 5:
        if f[-2] <= valid_split_time and f[-2] > train_split_time:
            start_num += 1
            valid_start_fact.add(f)
        if f[-1] <= valid_split_time and f[-1] > train_split_time:
            end_num += 1
            valid_end_fact.add(f)          
print([start_num, end_num])

print('test start end fact num')
start_num, end_num = 0, 0
test_start_fact = set()
test_end_fact = set()
for f in test_set:
    if len(f) == 5:
        if f[-2] > valid_split_time:
            start_num += 1
            test_start_fact.add(f)
        if f[-1] < sorted_all_times[-1] and f[-1] > valid_split_time:
            end_num += 1
            test_end_fact.add(f)
print([start_num, end_num])

# check train_valid_test 泄漏
# stamp knowledge
train_stamp_facts = set([f for f in train_set if len(f) == 4])
valid_stamp_facts = set([f for f in valid_set if len(f) == 4])
test_stamp_facts = set([f for f in test_set if len(f) == 4])

print("stamp train & valid")
print(len(train_stamp_facts&valid_stamp_facts))

print("stamp valid & test")
print(len(test_stamp_facts&valid_stamp_facts))

print("stamp train & test")
print(len(train_stamp_facts&test_stamp_facts))

# span knowledge
print("start train & valid")
print(len(train_start_fact&valid_start_fact))

print("start train & test")
print(len(train_start_fact&test_start_fact))

print("start test & valid")
print(len(test_start_fact&valid_start_fact))

print("end train & valid")
print(len(train_end_fact&valid_end_fact))

print("end train & test")
print(len(train_end_fact&test_end_fact))

print("end test & valid")
print(len(test_end_fact&valid_end_fact))

In [ ]:
pickle.dump([train_split_time, valid_split_time, train_set, valid_set, test_set, node_2_keys], open('ICEWSWiki_e60k_f5700k_train_valid_test.pkl', 'wb'))